PROGRMACIÓN ORIENTADA A OBJETOS

**DESCARGAR REPORTES SEMANALES UBER**

In [2]:
import pdfplumber
import re
import os
import glob
import pandas as pd

class UberReportExtractor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.full_text = ""
        self.page1_text = ""
        self.texto_plano = "" 
        self.data = {
            "driver_name": "Roman Yair Ortega",
            "weekly_period": "Not found",
            "total_trips": 0,
            "connected_hours_decimal": 0.0,
            "gross_amount": 0.0,
            "tips_amount": 0.0,
            "incentives_amount": 0.0,
            "taxes_amount": 0.0
        }

    def load_pdf(self):
        try:
            with pdfplumber.open(self.file_path) as pdf:
                for index, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text:
                        clean_text = text.replace("\u0000", "")
                        self.full_text += clean_text + "\n"
                        if index == 0:
                            self.page1_text = clean_text
            
            # Texto aplastado para forzar la adyacencia
            self.texto_plano = " ".join(self.full_text.split())
            return True
        except Exception as e:
            print(f"❌ Error loading PDF: {e}")
            return False

    def parse_data(self):
        # 1. Fechas 
        match_date = re.search(r"(\d{2}/\d{2}/\d{4}).*?(\d{2}/\d{2}/\d{4})", self.page1_text)
        if match_date: 
            self.data["weekly_period"] = f"Del {match_date.group(1)} al {match_date.group(2)}"
        else:
            match_date_alt = re.search(r"(\d{1,2}\s+[a-z]{3}\s+\d{4}.*?-.*?\d{4})", self.page1_text, re.IGNORECASE)
            if match_date_alt:
                self.data["weekly_period"] = re.sub(r"\s+\d{1,2}\s+[ap]\.\s+m\.", "", match_date_alt.group(1)).strip()

        # 2. Viajes
        match_trips = re.search(r"viajes completados[^\d]*(\d+)", self.full_text, re.IGNORECASE)
        if match_trips: self.data["total_trips"] = int(match_trips.group(1))
            
        # 3. Horas trabajadas
        
        match_time = re.search(r"Tiempo de trabajo efectivo.*?(\d+)\s*h.*?(\d+)\s*m", self.texto_plano, re.IGNORECASE)
        if match_time:
            self.data["connected_hours_decimal"] = round(int(match_time.group(1)) + (int(match_time.group(2)) / 60.0), 2)
        # =========================================================
        # 💰 CONVERSOR DE DINERO 
        # =========================================================
        def to_float(val_str):
            val_str = re.sub(r'[^\d\.,]', '', val_str)
            if not val_str: return 0.0
            if ',' in val_str and '.' in val_str:
                if val_str.rfind(',') > val_str.rfind('.'): 
                    val_str = val_str.replace('.', '').replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            elif ',' in val_str:
                if len(val_str) > 3 and val_str[-3] == ',':
                    val_str = val_str.replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            return float(val_str)

        # 🌟 EL CANDADO DE ADYACENCIA ESTRICTA (\s+)
        # Obliga a que el número esté PEGADO a la frase, ignorando subtítulos engañosos.
        # Soporta los nombres nuevos y viejos de Uber.
        
        match_gross = re.search(r"(?:Importe bruto que has generado|Monto bruto generado)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_gross: self.data["gross_amount"] = to_float(match_gross.group(1))

        match_tips = re.search(r"(?:Propinas dadas por los usuarios|Propinas otorgadas)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_tips: self.data["tips_amount"] = to_float(match_tips.group(1))

# 6. Incentivos (Agregamos "Otras ganancias" al diccionario de Uber)
        match_incentives = re.search(r"(?:Otras ganancias|Ganancias adicionales|promoción y fomento)\s*\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_incentives: self.data["incentives_amount"] = to_float(match_incentives.group(1))

        match_taxes = re.search(r"Impuestos\s*(?:-?\s*\$?|-)\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_taxes: self.data["taxes_amount"] = -abs(to_float(match_taxes.group(1)))


# ==========================================
# EL ADMINISTRADOR (Genera el CSV)
# ==========================================
class UberDatasetManager:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.all_reports_data = []
        self.pdf_files = []

    def find_pdfs(self):
        self.pdf_files = glob.glob(os.path.join(self.folder_path, "*.pdf"))
        print(f"🔍 Found {len(self.pdf_files)} PDF files in '{self.folder_path}'.")
        return len(self.pdf_files) > 0

    def process_all_reports(self):
        print("⚙️ Processing reports...")
        for file_path in self.pdf_files:
            extractor = UberReportExtractor(file_path)
            if extractor.load_pdf():
                extractor.parse_data()
                self.all_reports_data.append(extractor.data)

    def get_dataframe(self):
        return pd.DataFrame(self.all_reports_data) if self.all_reports_data else None


# ==========================================
# 3. EJECUCIÓN (La Magia Final)
# ==========================================
TARGET_FOLDER = "Descargas_Uber_Marzo_2025" 

dataset_maker = UberDatasetManager(TARGET_FOLDER)

if dataset_maker.find_pdfs():
    dataset_maker.process_all_reports()
    df_uber = dataset_maker.get_dataframe()
    
    # Filtramos las semanas vacías
    df_uber_limpio = df_uber[df_uber["total_trips"] > 0].copy()
    
    # 🌟 MÁSCARAS VISUALES (Dinero y Horas)
    formato_moneda = "${:,.2f}"
    formato_horas = "{:.2f} hrs"
    
    # Le decimos a Pandas cómo pintar cada columna
    df_estilizado = df_uber_limpio.style.format({
        'gross_amount': formato_moneda,
        'tips_amount': formato_moneda,
        'incentives_amount': formato_moneda,
        'taxes_amount': formato_moneda,
        'connected_hours_decimal': formato_horas
    })
    
    print("\n✅ DATASET COMPLETADO Y FORMATEADO:")
    display(df_estilizado)

🔍 Found 6 PDF files in 'Descargas_Uber_Marzo_2025'.
⚙️ Processing reports...


KeyboardInterrupt: 

**BOOT EXTRACTOR VIAJES HTML PAGINA UBER**

In [1]:
import json
import pandas as pd
from playwright.sync_api import sync_playwright
import os

# =========================================================================
# ⚠️ AVISO LEGAL / DISCLAIMER PARA USUARIOS EXTERNOS
# =========================================================================
# Este script es una herramienta de extracción de datos personales para uso 
# estrictamente individual. El autor no se hace responsable del mal uso, 
# la pérdida de acceso a la cuenta o cualquier violación a los términos de 
# servicio de Uber. Úselo bajo su propio riesgo y discreción.
# =========================================================================

viajes_acumulados = []
# Carpeta donde se guardarán tus credenciales (Cookies)
SESSION_FILE = "uber_session/gs_session.json"

def interceptar_trafico(response):
    global viajes_acumulados
    if "getWebActivityFeed" in response.url and response.status == 200:
        try:
            datos_json = response.json()
            nuevos_viajes = datos_json.get('data', {}).get('activities', [])
            if nuevos_viajes:
                viajes_acumulados.extend(nuevos_viajes)
                print(f"✅ ¡Pescados {len(nuevos_viajes)} viajes! (Total: {len(viajes_acumulados)})")
        except:
            pass

def ejecutar_bot():
    if not os.path.exists("uber_session"):
        os.makedirs("uber_session")

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        
        # Intentamos cargar la sesión previa si existe
        if os.path.exists(SESSION_FILE):
            print("🔐 Cargando sesión existente... (No deberías necesitar login)")
            context = browser.new_context(storage_state=SESSION_FILE)
        else:
            print("🆕 No se encontró sesión previa. Deberás loguearte esta vez.")
            context = browser.new_context()

        page = context.new_page()
        page.on("response", interceptar_trafico)

        print("🌐 Entrando a Uber Actividades...")
        page.goto("https://drivers.uber.com/earnings/activities")

        print("\n💡 INSTRUCCIONES:")
        print("1. Si la sesión expiró, inicia sesión una última vez.")
        print("2. Navega por tus semanas para capturar los datos.")
        print("3. Al terminar, presiona ENTER en esta terminal.")

        input("\n⏳ PRESIONA 'ENTER' PARA GUARDAR DATOS Y ACTUALIZAR SESIÓN...")

        # 💾 GUARDAR SESIÓN: Esto evita el login la próxima vez
        context.storage_state(path=SESSION_FILE)
        print(f"✔️ Sesión guardada en {SESSION_FILE}")

        if viajes_acumulados:
            df = pd.DataFrame(viajes_acumulados)
            df.to_csv("viajes_uber_bruto.csv", index=False)
            with open("viajes_uber_detallados.json", "w", encoding="utf-8") as f:
                json.dump(viajes_acumulados, f, ensure_ascii=False, indent=4)
            print(f"🏆 Proceso finalizado. {len(df)} viajes guardados.")
        
        browser.close()

if __name__ == "__main__":
    ejecutar_bot()

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.